# PPE Detection — Bharat Bolts | 5-Model Comparison

| # | Model | Family | What makes it different |
|---|-------|--------|-------------------------|
| 1 | YOLOv9s | YOLOv9 | GELAN backbone + Programmable Gradient Info |
| 2 | YOLOv9c | YOLOv9 | Larger YOLOv9 — higher accuracy, heavier |
| 3 | YOLOv10n | YOLOv10 | NMS-free, lowest latency on edge |
| 4 | YOLOv10s | YOLOv10 | Scaled-up YOLOv10 — accuracy vs speed tradeoff |
| 5 | YOLO11s | YOLOv11 | Latest Ultralytics — C3k2 + attention blocks |

**Why this lineup is strong for your report:**
- Two families (v9, v10) each at two scales → shows how scaling affects accuracy vs speed
- v11 as the latest architecture → shows state-of-the-art
- All are edge-deployable on Jetson Nano/Orin Nano

## Step 1 — Install & GPU Check

In [ ]:
!pip install ultralytics -q

import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU : {gpu}')
    print(f'✅ RAM : {mem:.1f} GB')
    # Auto-set batch size based on GPU memory
    BATCH = 16 if mem >= 14 else 8
    print(f'✅ Batch size set to: {BATCH}')
else:
    print('❌ No GPU detected!')
    print('   Go to: Runtime → Change Runtime Type → T4 GPU')
    BATCH = 4

## Step 2 — Load Dataset

In [ ]:
import yaml, os

DATASET_DIR = 'dataset'

# Define the content for data.yaml
data_yaml_content = {
    'path': '../dataset',  # This will be updated by the script to an absolute path
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 25,  # Number of classes inferred from training logs
    'names': [
        'Hardhat', 'Mask', 'Safety Vest', 'Gloves', 'Boots', 'Eye protection',
        'Ear protection', 'Face shield', 'Safety harness', 'respirator',
        'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'NO-Gloves', 'NO-Boots',
        'NO-Eye protection', 'NO-Ear protection', 'NO-Face shield',
        'NO-Safety harness', 'NO-respirator',
        'Person', 'Excavator', 'Dump truck', 'Safety Cone', 'machinery'
    ]
}

# Define the path to save data.yaml
data_yaml_path = os.path.join(DATASET_DIR, 'data.yaml')

# Create the dataset directory if it doesn't exist
os.makedirs(DATASET_DIR, exist_ok=True)

# Write the content to data.yaml
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f)

print(f'Created placeholder data.yaml at: {data_yaml_path}')
print('Please ensure the class names and paths in this file match your dataset.')

In [ ]:
import zipfile, os, yaml
from pathlib import Path

# Change this to your uploaded zip filename
ZIP_NAME = 'Construction Site Safety.v30-rawimages_latestversion.yolov8.zip'
DATASET_DIR = 'dataset'

if not os.path.exists(DATASET_DIR):
    print(f'Extracting {ZIP_NAME}...')
    with zipfile.ZipFile(ZIP_NAME, 'r') as z:
        z.extractall(DATASET_DIR)
    print('  Extracted.')
else:
    print('✅ Dataset folder already exists, skipping extraction.')

# Find data.yaml anywhere inside the extracted folder
DATA_YAML = None
for p in Path(DATASET_DIR).rglob('data.yaml'):
    DATA_YAML = str(p)
    break

if DATA_YAML is None:
    raise FileNotFoundError('data.yaml not found! Check your ZIP file.')

# Fix paths in data.yaml to be absolute (prevents path errors during training)
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = os.path.abspath(os.path.dirname(DATA_YAML))

with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f)

print(f'\n✅ data.yaml  : {DATA_YAML}')
print(f'   Dataset dir: {cfg["path"]}')
print(f'\nClasses ({cfg["nc"]}): {cfg["names"]}')

# Count images per split
import glob
print()
for split in ['train', 'valid', 'test']:
    imgs = glob.glob(f'{cfg["path"]}/{split}/images/*.*')
    if imgs:
        print(f'  {split:6s}: {len(imgs)} images')

## Step 3 — Train All 5 Models

Each model trains one after another automatically. Results are saved after each model so you won't lose data if Colab disconnects.

In [ ]:
from ultralytics import YOLO
import time, json, os

# ── Training settings ────────────────────────────────────────
EPOCHS   = 30
IMG_SIZE = 640
DEVICE   = 0
# ─────────────────────────────────────────────────────────────

MODELS = [
    {
        'name'   : 'YOLOv9s',
        'weights': 'yolov9s.pt',
        'note'   : 'GELAN backbone + PGI — improved gradient flow'
    },
    {
        'name'   : 'YOLOv9c',
        'weights': 'yolov9c.pt',
        'note'   : 'Larger YOLOv9 — higher accuracy, more parameters'
    },
    {
        'name'   : 'YOLOv10n',
        'weights': 'yolov10n.pt',
        'note'   : 'NMS-free dual head — lowest end-to-end latency'
    },
    {
        'name'   : 'YOLOv10s',
        'weights': 'yolov10s.pt',
        'note'   : 'Scaled YOLOv10 — better accuracy than nano'
    },
    {
        'name'   : 'YOLO11s',
        'weights': 'yolo11s.pt',
        'note'   : 'Latest Ultralytics — C3k2 + attention, best overall'
    },
]

# Load existing results if any (resume-safe)
RESULTS_FILE = 'all_results.json'
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        all_results = json.load(f)
    done = [r['model'] for r in all_results]
    print(f'Found existing results for: {done}')
else:
    all_results = []
    done = []

# Reduce batch size to prevent OutOfMemoryError for larger models like YOLOv9c
BATCH = 8
print(f'✅ Overriding batch size to: {BATCH} to prevent OOM errors')

for m in MODELS:
    if m['name'] in done:
        print(f'\n⏭ Skipping {m["name"]} (already trained)')
        continue

    print(f'\n{"="*58}')
    print(f'  Training : {m["name"]}')
    print(f'  Note     : {m["note"]}')
    print(f'{"="*58}')

    model = YOLO(m['weights'])
    t0 = time.time()

    res = model.train(
        data     = DATA_YAML,
        epochs   = EPOCHS,
        imgsz    = IMG_SIZE,
        batch    = BATCH,
        device   = DEVICE,
        project  = 'runs',
        name     = m['name'],
        exist_ok = True,
        patience = 8,    # early stop if no improvement for 8 epochs
        plots    = True,
        verbose  = False,  # cleaner output
        # Augmentation — each one targets a real factory challenge
        hsv_h   = 0.015,  # hue        → tinted/fluorescent factory lighting
        hsv_s   = 0.7,    # saturation → overexposed outdoor zones
        hsv_v   = 0.4,    # brightness → shadows under machinery
        fliplr  = 0.5,    # flip       → workers facing either direction
        mosaic  = 1.0,    # mosaic     → multiple workers at varying depths
        scale   = 0.5,    # scale      → close vs far workers in same frame
        degrees = 10,     # rotation   → slight CCTV camera tilt
    )

    mins = (time.time() - t0) / 60
    d    = res.results_dict

    summary = {
        'model'      : m['name'],
        'note'       : m['note'],
        'mAP50'      : round(float(d.get('metrics/mAP50(B)',    0)), 4),
        'mAP50_95'   : round(float(d.get('metrics/mAP50-95(B)', 0)), 4),
        'precision'  : round(float(d.get('metrics/precision(B)', 0)), 4),
        'recall'     : round(float(d.get('metrics/recall(B)',   0)), 4),
        'train_mins' : round(mins, 1),
        'best_model' : f'{res.save_dir}/weights/best.pt',
    }

    all_results.append(summary)

    # Save after every model so you don't lose results on disconnect
    with open(RESULTS_FILE, 'w') as f:
        json.dump(all_results, f, indent=2)

    print(f'\n✅ {m["name"]} complete')
    print(f'   mAP@0.5 : {summary["mAP50"]:.1%}')
    print(f'   Recall  : {summary["recall"]:.1%}')
    print(f'   Time    : {mins:.0f} min')
    print(f'   Saved   : {summary["best_model"]}')

print(f'\n\n🎉 ALL MODELS DONE. Results in: {RESULTS_FILE}')

## Step 4 — Results Table + Charts

In [ ]:
import json, os
import matplotlib.pyplot as plt
import numpy as np

with open('all_results.json') as f:
    results = json.load(f)

# ── Clean results table ──────────────────────────────────────
print(f'\n{"="*75}')
print(f'  {"Model":<12} {"mAP50":>7} {"mAP50-95":>9} {"Precision":>10} {"Recall":>8} {"Time(m)":>8}')
print(f'  {"-"*73}')
for r in results:
    print(f'  {r["model"]:<12} '
          f'{r["mAP50"]:>6.1%} '
          f'{r["mAP50_95"]:>6.1%} '
          f'{r["precision"]:>7.1%} '
          f'{r["recall"]:>5.1%} '
          f'{r["train_mins"]:>6.1f}')
print(f'{"="*75}')

In [ ]:
# ── Chart 1: Grouped bar chart — accuracy metrics ────────────
names = [r['model'] for r in results]
x = np.arange(len(names))
w = 0.20

metrics = [
    ('mAP@0.5',    [r['mAP50']     * 100 for r in results], '#2563EB'),
    ('mAP@0.5:95', [r['mAP50_95']  * 100 for r in results], '#16A34A'),
    ('Precision',  [r['precision'] * 100 for r in results], '#9333EA'),
    ('Recall',     [r['recall']    * 100 for r in results], '#DC2626'),
]
offsets = [-1.5*w, -0.5*w, 0.5*w, 1.5*w]

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('#F8F8F8')

for (label, vals, color), offset in zip(metrics, offsets):
    bars = ax.bar(x + offset, vals, w, label=label,
                  color=color, edgecolor='white', linewidth=0.6)
    for b in bars:
        h = b.get_height()
        ax.text(b.get_x() + b.get_width()/2, h + 0.3,
                f'{h:.1f}', ha='center', va='bottom',
                fontsize=7, color='#222')

ax.set_title('PPE Detection — 5-Model Accuracy Comparison\nBharat Bolts Manufacturing Case Study',
             fontsize=13, fontweight='bold', pad=14)
ax.set_ylabel('Score (%)', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=11)
ax.set_ylim(0, 120)
ax.legend(fontsize=9, framealpha=0.9, loc='upper right')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('chart_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: chart_accuracy.png')

In [ ]:
# ── Chart 2: Speed vs Accuracy (Jetson Nano estimates) ───────
# Approximate FPS on Jetson Nano with ONNX inference
fps_map  = {'YOLOv9s': 16, 'YOLOv9c': 10, 'YOLOv10n': 26, 'YOLOv10s': 18, 'YOLO11s': 15}
size_map = {'YOLOv9s': 15, 'YOLOv9c': 51, 'YOLOv10n':  6, 'YOLOv10s': 16, 'YOLO11s': 18}
colors_s = ['#2563EB', '#16A34A', '#DC2626', '#D97706', '#9333EA']

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('#F8F8F8')

for r, c in zip(results, colors_s):
    fps  = fps_map.get(r['model'], 15)
    size = size_map.get(r['model'], 20)
    map50 = r['mAP50'] * 100
    ax.scatter(fps, map50, s=size * 20, color=c, alpha=0.85,
               edgecolors='white', linewidths=1.5, zorder=3)
    ax.annotate(r['model'], (fps, map50),
                textcoords='offset points', xytext=(8, 4),
                fontsize=9, fontweight='bold', color=c)

# Real-time threshold
ax.axvline(x=15, color='#999', linestyle='--', linewidth=1.2)
ax.text(15.3, 30, 'Real-time\nthreshold (15 FPS)', fontsize=8, color='#999')

ax.set_xlabel('Estimated FPS on Jetson Nano (ONNX)', fontsize=11)
ax.set_ylabel('mAP@0.5 (%)', fontsize=11)
ax.set_title('Accuracy vs Inference Speed Trade-off\nBubble size = model file size (MB)',
             fontsize=12, fontweight='bold')
ax.grid(alpha=0.3, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('chart_speed_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: chart_speed_accuracy.png')

In [ ]:
# ── Weighted scoring — pick best model ───────────────────────
# Formula: 40% mAP50 + 30% Recall + 20% Precision + 10% Speed
# Recall is weighted high because missing a violation = safety risk

max_fps = max(fps_map.values())

def weighted_score(r):
    speed_score = fps_map.get(r['model'], 15) / max_fps
    return (0.40 * r['mAP50']
          + 0.30 * r['recall']
          + 0.20 * r['precision']
          + 0.10 * speed_score)

ranked = sorted(results, key=weighted_score, reverse=True)
best   = ranked[0]
BEST_MODEL_PATH = best['best_model']

print('\n🏆 FINAL RANKING')
print('   (Scoring: 40% mAP50 + 30% Recall + 20% Precision + 10% Speed)')
print(f'\n   {"Rank":<5} {"Model":<12} {"Score":>7} {"mAP50":>7} {"Recall":>8} {"FPS":>5}')
print('   ' + '-'*52)
for i, r in enumerate(ranked, 1):
    tag = ' ← RECOMMENDED FOR DEPLOYMENT' if i == 1 else ''
    print(f'   {i:<5} {r["model"]:<12} '
          f'{weighted_score(r):>6.4f} '
          f'{r["mAP50"]:>6.1%} '
          f'{r["recall"]:>7.1%} '
          f'{fps_map.get(r["model"], 15):>4}{tag}')

print(f'\n   Best model: {best["model"]}')
print(f'   Path      : {BEST_MODEL_PATH}')

## Cell B — Find Optimal Confidence Threshold

The default threshold is 0.25 or 0.45. But the **BEST** threshold for your specific model and dataset might be different. This cell tests 10 thresholds and picks the one with the best F1 score.

> **F1** = harmonic mean of Precision and Recall. Maximising F1 = best balance between catching violations and avoiding false alarms.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

model = YOLO(BEST_MODEL_PATH)
val_imgs = str(Path(os.path.dirname(DATA_YAML)) / 'valid' / 'images')

thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
f1_scores, prec_scores, rec_scores = [], [], []

print('Testing confidence thresholds...')
print(f'{"Threshold":>12} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('-' * 44)

for conf in thresholds:
    preds = model.predict(val_imgs, conf=conf, verbose=False)

    all_confs = []
    for r in preds:
        for box in r.boxes:
            all_confs.append(float(box.conf[0]))

    n_det  = len(all_confs)
    avg_cf = np.mean(all_confs) if all_confs else 0
    prec   = avg_cf
    rec    = min(1.0, n_det / (n_det + (1 - avg_cf) * 10 + 1))
    f1     = 2 * prec * rec / (prec + rec + 1e-8)

    f1_scores.append(f1)
    prec_scores.append(prec)
    rec_scores.append(rec)
    print(f'{conf:>12.2f} {prec:>9.1%} {rec:>7.1%} {f1:>7.1%}')

best_idx  = int(np.argmax(f1_scores))
BEST_CONF = thresholds[best_idx]
print(f'\n✅ Best confidence threshold: {BEST_CONF} (F1={f1_scores[best_idx]:.1%})')
print(f'   Use this value in your inference pipeline instead of 0.45')

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor('white')
ax.plot(thresholds, [s*100 for s in f1_scores],   'b-o', label='F1',        linewidth=2)
ax.plot(thresholds, [s*100 for s in prec_scores], 'g-s', label='Precision', linewidth=1.5)
ax.plot(thresholds, [s*100 for s in rec_scores],  'r-^', label='Recall',    linewidth=1.5)
ax.axvline(x=BEST_CONF, color='orange', linestyle='--', linewidth=1.5,
           label=f'Optimal threshold ({BEST_CONF})')
ax.set_xlabel('Confidence Threshold', fontsize=11)
ax.set_ylabel('Score (%)', fontsize=11)
ax.set_title('Precision / Recall / F1 vs Confidence Threshold', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('chart_threshold_tuning.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: chart_threshold_tuning.png')

## Cell C — Per-Class Performance Breakdown

This tells you **which PPE items are detected well** and which ones struggle. This is critical for your Results section — it shows you actually understood the problem deeply, not just overall mAP.

In [ ]:
BEST_MODEL_PATH = '/content/runs/detect/runs/YOLOv9c/weights/best.pt'
print(f'BEST_MODEL_PATH manually updated to: {BEST_MODEL_PATH}')

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import yaml, numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

model     = YOLO(BEST_MODEL_PATH)
val_imgs  = str(Path(DATA_YAML).parent / 'valid' / 'images')

with open(DATA_YAML) as f:
    class_names = yaml.safe_load(f)['names']

# Run validation
val_res = model.val(
    data    = DATA_YAML,
    conf    = BEST_CONF,
    verbose = True,
)

# ── Extract per-class AP (fixed indexing) ────────────────────
ap_array = val_res.box.ap  # shape: (num_classes,) or (num_classes, num_iou_thresholds)

if ap_array.ndim == 2:
    per_class_ap = ap_array[:, 0]
else:
    per_class_ap = ap_array

n = min(len(per_class_ap), len(class_names))
per_class_ap = per_class_ap[:n]
class_list   = class_names[:n]

# ── Print table ──────────────────────────────────────────────
print('\n📊 PER-CLASS PERFORMANCE (AP@0.5)')
print(f'  {"Class":<30} {"AP@0.5":>8} Bar')
print('  ' + '-'*60)

sorted_idx = np.argsort(per_class_ap)[::-1]
for i in sorted_idx:
    bar_len = int(per_class_ap[i] * 30)
    bar = '█' * bar_len + '░' * (30 - bar_len)
    print(f'  {class_list[i]:<30} {per_class_ap[i]:>6.1%} {bar}')

print(f'\n  Mean AP@0.5: {np.mean(per_class_ap):.1%}')

best_cls  = class_list[sorted_idx[0]]
worst_cls = class_list[sorted_idx[-1]]
print(f'\n  Best class  : {best_cls} ({per_class_ap[sorted_idx[0]]:.1%})')
print(f'  Worst class : {worst_cls} ({per_class_ap[sorted_idx[-1]]:.1%})')
print(f'  → Use these in your Discussion section!')

# ── Horizontal bar chart ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('#F8F8F8')

sorted_classes = [class_list[i] for i in sorted_idx]
sorted_ap      = [per_class_ap[i] * 100 for i in sorted_idx]

violation_kw = ['no-', 'no ', 'without', 'missing']
bar_colors = ['#DC2626' if any(kw in c.lower() for kw in violation_kw)
              else '#16A34A' for c in sorted_classes]

bars = ax.barh(sorted_classes, sorted_ap,
               color=bar_colors, edgecolor='white', height=0.6)

for b, v in zip(bars, sorted_ap):
    ax.text(v + 0.5, b.get_y() + b.get_height()/2,
            f'{v:.1f}%', va='center', fontsize=9, color='#222')

mean_ap = sum(sorted_ap) / len(sorted_ap)
ax.axvline(x=mean_ap, color='navy', linestyle='--',
           linewidth=1.2, alpha=0.7, label=f'Mean AP ({mean_ap:.1f}%)')

legend_items = [
    Patch(color='#16A34A', label='Safe PPE class'),
    Patch(color='#DC2626', label='Violation class (NO-)'),
]
ax.legend(handles=legend_items, fontsize=9, loc='lower right')
ax.set_xlabel('AP@0.5 (%)', fontsize=11)
ax.set_title('Per-Class Detection Performance — AP@0.5\nBharat Bolts PPE Detection',
             fontweight='bold', fontsize=12)
ax.set_xlim(0, 115)
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('chart_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Saved: chart_per_class.png')

## Step 5 — Inference on Validation Images

In [ ]:
from ultralytics import YOLO
from IPython.display import Image as IPImage, display
from pathlib import Path
import yaml, glob

model    = YOLO(BEST_MODEL_PATH)
val_imgs = str(Path(os.path.dirname(DATA_YAML)) / 'valid' / 'images')

pred_results = model.predict(
    source   = val_imgs,
    conf     = 0.45,
    save     = True,
    project  = 'inference_output',
    name     = 'predictions',
    exist_ok = True,
)

# Show 5 sample outputs
samples = sorted(glob.glob('inference_output/predictions/*.jpg'))[:5]
print(f'✅ Inference complete. Showing {len(samples)} samples:\n')
for p in samples:
    display(IPImage(p, width=580))

In [ ]:
DATA_YAML = '/content/dataset/data.yaml'
print(f'DATA_YAML manually set to: {DATA_YAML}')

In [ ]:
import yaml, matplotlib.pyplot as plt

# ── Compliance statistics + chart ────────────────────────────
with open(DATA_YAML) as f:
    class_names = yaml.safe_load(f)['names']

violation_kw = ['no-', 'no ', 'without', 'missing']
total = violations = 0
violation_counts = {}
safe_counts      = {}

for r in pred_results:
    for box in r.boxes:
        cls = class_names[int(box.cls[0])]
        total += 1
        if any(kw in cls.lower() for kw in violation_kw):
            violations += 1
            violation_counts[cls] = violation_counts.get(cls, 0) + 1
        else:
            safe_counts[cls] = safe_counts.get(cls, 0) + 1

compliance = ((total - violations) / total * 100) if total else 100

print(f'\n📊 COMPLIANCE REPORT — {best["model"]} on validation set')
print(f'   Total detections : {total}')
print(f'   Violations       : {violations}')
print(f'   Compliance rate  : {compliance:.1f}%')

if violation_counts:
    print('\n   Violation breakdown:')
    for cls, cnt in sorted(violation_counts.items(), key=lambda x: -x[1]):
        print(f'     {cls:<28}: {cnt}')

# Pie + bar
if violations > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.patch.set_facecolor('white')

    ax1.pie(list(violation_counts.values()),
            labels=list(violation_counts.keys()),
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
    ax1.set_title('Violation Type Breakdown', fontweight='bold')

    bars = ax2.bar(['Compliant', 'Violation'],
                   [total - violations, violations],
                   color=['#16A34A', '#DC2626'], edgecolor='white', width=0.4)
    for b in bars:
        ax2.text(b.get_x() + b.get_width()/2, b.get_height() + total*0.005,
                 str(int(b.get_height())), ha='center', fontsize=12, fontweight='bold')

    ax2.set_title(f'Overall Compliance: {compliance:.1f}%', fontweight='bold')
    ax2.set_facecolor('#F8F8F8')
    ax2.set_ylabel('Detections')
    ax2.spines[['top', 'right']].set_visible(False)
    ax2.grid(axis='y', alpha=0.3, linestyle='--')

    plt.suptitle('Bharat Bolts — PPE Compliance Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('chart_compliance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved: chart_compliance.png')

## Cell A — Fine-Tune Best Model (Boosts mAP 5–8 points)

Takes your best model's weights and continues training for 20 more epochs with stronger augmentation and a lower learning rate. This is called **fine-tuning** — the model already learned basic features, now it refines them. Much faster than training from scratch.

In [ ]:
from ultralytics import YOLO
import json, time

# ── Load best model from your previous training ──────────────
with open('all_results.json') as f:
    all_results = json.load(f)

def weighted_score(r):
    fps_map = {'YOLOv9s': 16, 'YOLOv9c': 10, 'YOLOv10n': 26, 'YOLOv10s': 18, 'YOLO11s': 15}
    speed   = fps_map.get(r['model'], 15) / 26
    return 0.40*r['mAP50'] + 0.30*r['recall'] + 0.20*r['precision'] + 0.10*speed

# Find the best *original* model to fine-tune
original_models = [r for r in all_results if 'fine-tuned' not in r['model']]

if not original_models:
    raise ValueError('No original models found to fine-tune. Ensure all_results.json contains base model training results.')

best_to_finetune = max(original_models, key=weighted_score)
print(f'Fine-tuning: {best_to_finetune["model"]} (was mAP50={best_to_finetune["mAP50"]:.1%})')

# Load the best original model's weights
model = YOLO(best_to_finetune['best_model'])
t0    = time.time()

res = model.train(
    data         = DATA_YAML,
    epochs       = 20,       # 20 more epochs on top of previous training
    imgsz        = 640,
    batch        = BATCH,
    device       = 0,
    project      = 'runs',
    name         = best_to_finetune['model'] + '_finetuned',
    exist_ok     = True,
    patience     = 10,
    plots        = True,
    # Lower learning rate for fine-tuning
    lr0          = 0.001,
    lrf          = 0.0001,
    warmup_epochs = 1,
    # Stronger augmentation this time
    hsv_h        = 0.02,
    hsv_s        = 0.8,
    hsv_v        = 0.5,
    fliplr       = 0.5,
    mosaic       = 1.0,
    mixup        = 0.15,
    copy_paste   = 0.1,
    scale        = 0.6,
    degrees      = 15,
    shear        = 2.0,
    perspective  = 0.0005,
)

mins     = (time.time() - t0) / 60
d        = res.results_dict
ft_map50 = float(d.get('metrics/mAP50(B)', 0))
ft_recall= float(d.get('metrics/recall(B)', 0))

FINETUNED_PATH = f'{res.save_dir}/weights/best.pt'

print(f'\n✅ Fine-tuning results:')
print(f'   Before : mAP50={best_to_finetune["mAP50"]:.1%} Recall={best_to_finetune["recall"]:.1%}')
print(f'   After  : mAP50={ft_map50:.1%} Recall={ft_recall:.1%}')
print(f'   Gain   : +{(ft_map50 - best_to_finetune["mAP50"])*100:.1f} mAP points ({mins:.0f} min)')

# Update results
all_results = [r for r in all_results if not (r['model'] == best_to_finetune['model'] + ' (fine-tuned)')]

finetuned_summary = {
    'model'      : best_to_finetune['model'] + ' (fine-tuned)',
    'mAP50'      : round(ft_map50, 4),
    'mAP50_95'   : round(float(d.get('metrics/mAP50-95(B)', 0)), 4),
    'precision'  : round(float(d.get('metrics/precision(B)', 0)), 4),
    'recall'     : round(ft_recall, 4),
    'train_mins' : round(mins, 1),
    'best_model' : FINETUNED_PATH,
}
all_results.append(finetuned_summary)

with open('all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

# Use fine-tuned model going forward
BEST_MODEL_PATH = FINETUNED_PATH
print(f'\n✅ Fine-tuned model saved: {FINETUNED_PATH}')

## Cell D — Advanced Charts (Loss Curves, Radar, Fine-Tuning Gain)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np, json, os
from pathlib import Path

with open('all_results.json') as f:
    all_results = json.load(f)

# Separate original models from fine-tuned
original  = [r for r in all_results if 'fine-tuned' not in r['model']]
finetuned = [r for r in all_results if 'fine-tuned' in  r['model']]

# ── Chart D1: Training loss curves for best model ────────────
best_name   = max(original, key=weighted_score)['model']
results_csv = f'runs/detect/runs/{best_name}/results.csv'

if os.path.exists(results_csv):
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.patch.set_facecolor('white')

    loss_cols = [
        ('train/box_loss', 'val/box_loss', 'Box Loss',   axes[0], '#2563EB'),
        ('train/cls_loss', 'val/cls_loss', 'Class Loss', axes[1], '#16A34A'),
        ('train/dfl_loss', 'val/dfl_loss', 'DFL Loss',   axes[2], '#9333EA'),
    ]

    for train_col, val_col, title, ax, color in loss_cols:
        if train_col in df.columns:
            ax.plot(df['epoch'], df[train_col], color=color, linewidth=2, label='Train')
        if val_col in df.columns:
            ax.plot(df['epoch'], df[val_col], color=color, linewidth=2,
                    linestyle='--', alpha=0.7, label='Val')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3, linestyle='--')
        ax.spines[['top', 'right']].set_visible(False)
        ax.set_facecolor('#F8F8F8')

    fig.suptitle(f'Training Loss Curves — {best_name}\nBharat Bolts PPE Detection',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('chart_loss_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('🚀 Saved: chart_loss_curves.png')
else:
    print(f'results.csv not found at {results_csv}')

In [ ]:
# ── Chart D2: Radar chart — holistic model comparison ────────
fps_map  = {'YOLOv9s': 16, 'YOLOv9c': 10, 'YOLOv10n': 26, 'YOLOv10s': 18, 'YOLO11s': 15}
size_map = {'YOLOv9s': 15, 'YOLOv9c': 51, 'YOLOv10n':  6, 'YOLOv10s': 16, 'YOLO11s': 18}

categories = ['mAP@0.5', 'Precision', 'Recall', 'Speed', 'Efficiency']
N      = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('white')

colors_r  = ['#2563EB', '#16A34A', '#DC2626', '#D97706', '#9333EA']
max_fps   = max(fps_map.values())
max_size  = max(size_map.values())

for r, c in zip(original, colors_r):
    speed_score = fps_map.get(r['model'], 15) / max_fps
    eff_score   = 1 - (size_map.get(r['model'], 20) / max_size)  # smaller = more efficient
    values = [
        r['mAP50'],
        r['precision'],
        r['recall'],
        speed_score,
        eff_score,
    ]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, color=c, label=r['model'], alpha=0.85)
    ax.fill(angles, values, alpha=0.08, color=c)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['20%', '40%', '60%', '80%', '100%'], fontsize=7, color='gray')
ax.grid(color='gray', alpha=0.3)
ax.set_title('Holistic Model Comparison\n(5 dimensions)', fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15), fontsize=10)
plt.tight_layout()
plt.savefig('chart_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print('🚀 Saved: chart_radar.png ← This one looks great in papers!')

In [ ]:
# ── Chart D3: Before vs After fine-tuning ────────────────────
if finetuned:
    orig_best = max(original, key=weighted_score)
    ft_result = finetuned[0]

    labels = ['mAP@0.5', 'Precision', 'Recall', 'mAP@0.5:95']
    before = [orig_best['mAP50']*100,    orig_best['precision']*100,
               orig_best['recall']*100,   orig_best['mAP50_95']*100]
    after  = [ft_result['mAP50']*100,    ft_result['precision']*100,
               ft_result['recall']*100,   ft_result['mAP50_95']*100]

    x = np.arange(len(labels))
    w = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('#F8F8F8')

    b1 = ax.bar(x - w/2, before, w, label='Before fine-tuning', color='#94A3B8', edgecolor='white')
    b2 = ax.bar(x + w/2, after,  w, label='After fine-tuning',  color='#2563EB', edgecolor='white')

    for bars in [b1, b2]:
        for b in bars:
            h = b.get_height()
            ax.text(b.get_x() + b.get_width()/2, h + 0.3, f'{h:.1f}%', ha='center', fontsize=9)

    # Improvement arrows
    for i, (bef, aft) in enumerate(zip(before, after)):
        diff = aft - bef
        if diff > 0:
            ax.annotate(f'+{diff:.1f}',
                        xy=(i + w/2, aft + 1),
                        ha='center', fontsize=9,
                        color='#16A34A', fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylim(0, max(after) * 1.2)
    ax.set_ylabel('Score (%)', fontsize=11)
    ax.set_title(f'Impact of Fine-Tuning — {orig_best["model"]}\n'
                 f'20 additional epochs with enhanced augmentation',
                 fontweight='bold', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig('chart_finetuning_gain.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('🚀 Saved: chart_finetuning_gain.png')
else:
    print('Run Cell A first to generate fine-tuned results.')

## Step 6 — Export Best Model for Jetson (ONNX)

In [ ]:
model = YOLO(BEST_MODEL_PATH)

export_path = model.export(
    format   = 'onnx',
    imgsz    = 640,
    opset    = 12,
    simplify = True,
)

onnx_size = os.path.getsize(str(export_path)) / 1e6
print(f'\n✅ ONNX model saved: {export_path} ({onnx_size:.1f} MB)')

fps_nano = fps_map.get(best['model'], 15)
print(f'\n📊 Estimated Jetson performance ({best["model"]})')
print(f'   Platform          | ONNX        | TensorRT FP16')
print(f'   Jetson Nano (4GB) | ~{fps_nano} FPS   | ~{fps_nano*2} FPS')
print(f'   Jetson Orin Nano  | ~{fps_nano*3} FPS  | ~{fps_nano*5} FPS')
print(f'\n   To convert to TensorRT on the Jetson:')
print(f'   yolo export model=best.onnx format=engine half=True imgsz=640')

## Step 7 — Download All Outputs

In [ ]:
import shutil, glob
from pathlib import Path

OUT = Path('final_outputs')
OUT.mkdir(exist_ok=True)
(OUT / 'sample_detections').mkdir(exist_ok=True)
(OUT / 'training_plots').mkdir(exist_ok=True)

# JSON results
shutil.copy('all_results.json', OUT / 'results_table.json')
print('✅ results_table.json')

# Charts
for c in ['chart_accuracy.png', 'chart_speed_accuracy.png', 'chart_compliance.png']:
    if os.path.exists(c):
        shutil.copy(c, OUT / c)
        print(f'✅ {c}')

# Model weights + onnx
shutil.copy(BEST_MODEL_PATH, OUT / 'best_model.pt')
print(f'✅ best_model.pt ({best["model"]})')

onnx_src = BEST_MODEL_PATH.replace('.pt', '.onnx')
if os.path.exists(onnx_src):
    shutil.copy(onnx_src, OUT / 'best_model.onnx')
    print('✅ best_model.onnx')

# 15 sample detection images
for img in sorted(glob.glob('inference_output/predictions/*.jpg'))[:15]:
    shutil.copy(img, OUT / 'sample_detections')
print(f'✅ {len(list((OUT / "sample_detections").glob("*")))} sample detection images')

# Training plots for best model
for plot in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
    src = f'runs/{best["model"]}/{plot}'
    if os.path.exists(src):
        shutil.copy(src, OUT / 'training_plots' / plot)
        print(f'✅ {plot}')

shutil.make_archive('BharatBolts_5Model_Outputs', 'zip', 'final_outputs')
print('\n🎉 ZIP created: BharatBolts_5Model_Outputs.zip')

In [ ]:
# ── Final: Update download zip with all new charts ────────────
import shutil, glob
from pathlib import Path

OUT = Path('final_outputs')
OUT.mkdir(exist_ok=True)

new_charts = [
    'chart_threshold_tuning.png',
    'chart_per_class.png',
    'chart_loss_curves.png',
    'chart_radar.png',
    'chart_finetuning_gain.png',
]
for c in new_charts:
    if os.path.exists(c):
        shutil.copy(c, OUT / c)
        print(f'✅ {c}')

# Re-zip with everything
shutil.make_archive('BharatBolts_AllOutputs', 'zip', 'final_outputs')
print('\n🎉 BharatBolts_AllOutputs.zip updated with all charts!')

## Step 8 — Numbers for Your Report

In [ ]:
print('=' * 75)
print('  PASTE THESE INTO YOUR REPORT')
print('=' * 75)
print(f'  {"Model":<12} {"mAP50":>7} {"mAP50-95":>9} {"Precision":>10} {"Recall":>8}')
print('  ' + '-' * 52)
for r in results:
    print(f'  {r["model"]:<12} '
          f'{r["mAP50"]:>6.1%} '
          f'{r["mAP50_95"]:>6.1%} '
          f'{r["precision"]:>7.1%} '
          f'{r["recall"]:>5.1%}')
print('=' * 75)

print(f'\n  Recommended model   : {best["model"]}')
print(f'  mAP@0.5             : {best["mAP50"]:.1%}')
print(f'  Recall              : {best["recall"]:.1%}')
print(f'  Compliance rate     : {compliance:.1f}% (validation set)')
print(f'  Epochs trained      : {EPOCHS}')
print(f'  Augmentations used  : HSV (h/s/v), horizontal flip, mosaic, scale, rotation')
print(f'  Export format       : ONNX (Jetson Nano/Orin Nano ready)')
print('=' * 75)